In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os.path
from pathlib import Path
import pickle
import multiprocessing
from tqdm import tqdm

In [2]:
import import_ipynb

In [3]:
import DTW

In [4]:
import NWTW

The Cython extension is already loaded. To reload it, use:
  %reload_ext Cython


In [5]:
import FlexDTW

The Cython extension is already loaded. To reload it, use:
  %reload_ext Cython


In [6]:
import Parflex
import Parflex_gpu  # Stage-1 FlexDTW on GPU when available; same `parflex(C, ...)` API as Parflex

In [7]:
DATASET = 'train' # 'test'
VERSION = 'toy'

In [8]:
QUERY_LIST = Path(f'cfg_files/queries.{DATASET}.{VERSION}')

In [9]:
# SYSTEMS = ['dtw1', 'dtw2', 'dtw3', 'subseqdtw1', 'subseqdtw2', 'subseqdtw3', 'nwtw', 'flexdtw', 'parflex', 'cpu_parflex']

SYSTEMS = ['parflex', 'gpu_parflex']
# Change train or test in the parflex sweep: 
from parflex_config import L_VALUES as SPARSE_PARFLEX_L_VALUES
# To run with a constant L (single value) instead of the full L sweep, uncomment and set:

BENCHMARKS = ['matching', 'subseq_20', 'subseq_30', 'subseq_40', 'partialStart', 'partialEnd', 'partialOverlap', 
              'pre_5', 'pre_10', 'pre_20', 'post_5', 'post_10', 'post_20', 'prepost_5', 'prepost_10',
              'prepost_20']

In [10]:
features_root = Path('../ttmp/Chopin_Mazurkas_features')
FEAT_DIRS = {}

for benchmark in BENCHMARKS:
    if benchmark == 'partialOverlap':
        FEAT_DIRS[benchmark] = ([features_root/'partialStart', features_root/'partialEnd'])
    elif 'prepost' in benchmark:
        sec = benchmark.split('_')[-1]
        FEAT_DIRS[benchmark] = ([features_root/f'pre_{sec}', features_root/f'post_{sec}'])
    else:
        FEAT_DIRS[benchmark] = [features_root/f'{benchmark}', features_root/'original']

In [11]:
steps = {'dtw1': np.array([1,1,1,2,2,1]).reshape((-1,2)),
        'dtw2': np.array([1,1,1,2,2,1]).reshape((-1,2)),
        'dtw3': np.array([1,1,1,2,2,1]).reshape((-1,2)),
        'subseqdtw1': np.array([1,1,1,2,2,1]).reshape((-1,2)),
        'subseqdtw2': np.array([1,1,1,2,2,1]).reshape((-1,2)),
        'subseqdtw3': np.array([1,1,1,2,2,1]).reshape((-1,2)),
        'nwtw': 0, # transitions are specified in NWTW algorithm
        'flexdtw': np.array([1,1,1,2,2,1]).reshape((-1,2)), 
        'parflex': np.array([1,1,1,2,2,1]).reshape((-1,2)),
        'gpu_parflex': np.array([1,1,1,2,2,1]).reshape((-1,2)),
        'cpu_parflex': np.array([1,1,1,2,2,1]).reshape((-1,2)),
        'sparse_parflex': np.array([1,1,1,2,2,1]).reshape((-1,2))
        }
weights = {'dtw1': np.array([2,3,3]),
          'dtw2': np.array([1,1,1]),
          'dtw3': np.array([1,2,2]),
          'subseqdtw1': np.array([1,1,2]),
          'subseqdtw2': np.array([2,3,3]),
          'subseqdtw3': np.array([1,2,2]),
          'nwtw': 0, # weights are specified in NWTW algorithm
          'flexdtw': np.array([1.25,3,3]),
          'parflex': np.array([1.25,3.0,3.0]),
          'gpu_parflex': np.array([1.25,3.0,3.0]),
          'cpu_parflex': np.array([1.25,3.0,3.0]),
          'sparse_parflex': np.array([1.25,3.0,3.0])
          }
other_params = {
                'flexdtw': {'beta': 0.1}, 
                'parflex': {'beta': 0.1},
                'gpu_parflex': {'beta': 0.1},
                'cpu_parflex': {'beta': 0.1},
                'sparse_parflex': {'beta': 0.1}
               }

# Benchmarks

In [12]:
def get_outfile(outdir, benchmark, system, queryid, L=None):
    """L is used only for system=='sparse_parflex': output goes to .../sparse_parflex/L_<L>/queryid.pkl"""
    outpath = (outdir / benchmark / system)
    if system == 'sparse_parflex' and L is not None:
        outpath = outpath / f'L_{L}'
    outpath.mkdir(parents=True, exist_ok=True)
    outfile = (outpath / queryid).with_suffix('.pkl')
    return outfile

In [13]:
# Parflex: Parflex.ipynb / Parflex.py. GPU Stage-1: Parflex_gpu.py (`Parflex_gpu.parflex`).


In [14]:
def align_system(system, F1, F2, outfile, L=None):
    subseq = 'subseq' in system
    if system == 'parflex':
        C = 1 - FlexDTW.L2norm(F1).T @ FlexDTW.L2norm(F2)
        best_cost, wp = Parflex.parflex(C, steps=steps[system], weights=weights[system], beta=other_params[system]['beta'], L=other_params[system].get('L'))

    elif system == 'gpu_parflex':
        C = 1 - FlexDTW.L2norm(F1).T @ FlexDTW.L2norm(F2)
        best_cost, wp = Parflex_gpu.parflex(
            C,
            steps=steps[system],
            weights=weights[system],
            beta=other_params[system]['beta'],
            L=other_params[system].get('L'),
        )

    elif system == 'cpu_parflex':
        C = 1 - FlexDTW.L2norm(F1).T @ FlexDTW.L2norm(F2)
        best_cost, wp = cpu_parflex.parflex(C, steps=steps[system], weights=weights[system], beta=other_params[system]['beta'], L=other_params[system].get('L'))

    elif system == 'sparse_parflex':
        C = 1 - FlexDTW.L2norm(F1).T @ FlexDTW.L2norm(F2)
        best_cost, wp = Sparse_Parflex.parflex(C, steps=steps[system], weights=weights[system], beta=other_params[system]['beta'], L=L if L is not None else other_params[system].get('L'))
    
    elif system == 'flexdtw':
        L1 = F1.shape[1]
        L2 = F2.shape[1]
        buffer = min(L1, L2) * (1 - (1 - other_params[system]['beta']) * min(L1,L2) / max(L1, L2))
        C = 1 - FlexDTW.L2norm(F1).T @ FlexDTW.L2norm(F2) # cos distance metric 
        best_cost, wp, D, P, B, debug = FlexDTW.flexdtw(C, steps=steps[system], weights=weights[system], buffer=buffer)
        
    elif system == 'nwtw':
        downsample = 1
        C = 1 - NWTW.L2norm(F1)[:,0::downsample].T @ NWTW.L2norm(F2)[:,0::downsample] # cos distance metric
        optcost, wp, D, B = NWTW.NWTW_faster(C, gamma=0.346)
    else:
        downsample = 1
        if subseq and (F2.shape[1] < F1.shape[1]):
            C = 1 - DTW.L2norm(F2)[:,0::downsample].T @ DTW.L2norm(F1)[:,0::downsample] # cos distance metric
            wp = DTW.alignDTW(C, steps=steps[system], weights=weights[system], downsample=downsample, outfile=outfile, subseq=subseq)
            wp = wp[::-1,:]
        else:
            C = 1 - DTW.L2norm(F1)[:,0::downsample].T @ DTW.L2norm(F2)[:,0::downsample] # cos distance metric
            wp = DTW.alignDTW(C, steps=steps[system], weights=weights[system], downsample=downsample, outfile=outfile, subseq=subseq)
            
    if wp is not None:
        pickle.dump(wp, open(outfile, 'wb'))


In [15]:
def run_all_benchmarks(outdir):
    parts_batch = []
    queryids = []
    with open(QUERY_LIST, 'r') as f:
        for line in f:
            parts = line.strip().split(' ')
            assert len(parts) == 2
            queryid = os.path.basename(parts[0]) + '__' + os.path.basename(parts[1])
            
            if 'Czerny-Stefanska-1949_pid9086' in queryid:
                continue
            
            parts_batch.append(parts)
            queryids.append(queryid)
            
    for benchmark in tqdm(BENCHMARKS):
#         for i in range(len(parts_batch)):
#             run_benchmark(benchmark, FEAT_DIRS[benchmark][0], FEAT_DIRS[benchmark][1], parts_batch[i], outdir, queryids[i])
        run_benchmark_batch(benchmark, FEAT_DIRS[benchmark][0], FEAT_DIRS[benchmark][1], parts_batch, outdir, queryids, n_cores=None)

In [ ]:
import math
import threading
import multiprocessing.pool as mp_pool

# Serialize `gpu_parflex` work: the outer batch uses a thread pool; concurrent
# Parflex_gpu/CuPy calls on one device are unsafe or can OOM.
_gpu_parflex_lock = threading.Lock()

# NOTE: `multiprocessing.Pool` workers are daemonic by default, which prevents
# `cpu_parflex` (which itself spawns a Pool) from creating child processes.
# These helpers let us run `cpu_parflex` safely inside an outer pool.
class NonDaemonProcess(multiprocessing.Process):
    def _get_daemon(self):
        return False
    def _set_daemon(self, value):
        pass
    daemon = property(_get_daemon, _set_daemon)

class NonDaemonPool(mp_pool.Pool):
    Process = NonDaemonProcess

def _estimate_cpu_parflex_workers_from_lengths(L1, L2, chunk_length=None, max_workers=16):
    if chunk_length is None:
        chunk_length = cpu_parflex.DEFAULT_CHUNK_LENGTH
    hop = chunk_length - 1
    if hop <= 0:
        return 1
    n_chunks_1 = math.ceil((L1 - 1) / hop) if L1 > 1 else 1
    n_chunks_2 = math.ceil((L2 - 1) / hop) if L2 > 1 else 1
    total_chunks = n_chunks_1 * n_chunks_2
    return max(1, min(int(total_chunks), max_workers))

def _align_system_task(task):
    system, featfile1, featfile2, outfile, L = task
    F1 = np.load(featfile1)
    F2 = np.load(featfile2)
    # `align_system` writes the pickle directly to `outfile`.
    if system == 'gpu_parflex':
        with _gpu_parflex_lock:
            align_system(system, F1, F2, outfile, L)
    else:
        align_system(system, F1, F2, outfile, L)

def run_benchmark_batch(benchmark, featdir1, featdir2, parts_batch, outdir, queryids, n_cores=None):
    inputs = []
    assert len(parts_batch) == len(queryids)

    max_inner_workers = 1
    saw_cpu_parflex = False

    for i in range(len(parts_batch)):
        featfile1 = (featdir1 / parts_batch[i][0]).with_suffix('.npy')
        featfile2 = (featdir2 / parts_batch[i][1]).with_suffix('.npy')

        # Only need frame counts to size the outer pool.
        f1_m = np.load(featfile1, mmap_mode='r')
        f2_m = np.load(featfile2, mmap_mode='r')
        L1 = f1_m.shape[1]
        L2 = f2_m.shape[1]
        del f1_m, f2_m

        for system in SYSTEMS:
            if system == 'sparse_parflex':
                for L in SPARSE_PARFLEX_L_VALUES:
                    outfile = get_outfile(outdir, benchmark, system, queryids[i], L=L)
                    if not os.path.isfile(outfile):
                        inputs.append((system, featfile1, featfile2, outfile, L))
            else:
                outfile = get_outfile(outdir, benchmark, system, queryids[i])
                if not os.path.isfile(outfile):
                    inputs.append((system, featfile1, featfile2, outfile, None))
                    if system == 'cpu_parflex':
                        saw_cpu_parflex = True
                        est = _estimate_cpu_parflex_workers_from_lengths(L1, L2)
                        max_inner_workers = max(max_inner_workers, est)

    if not inputs:
        return

    cpu_total = os.cpu_count() or 1
    effective_cpus = max(1, cpu_total - 1)  # leave 1 core for the OS

    if saw_cpu_parflex:
        # Account for the outer worker itself (+1) and the inner Pool processes.
        outer_procs = effective_cpus // max(1, (max_inner_workers + 1))
    else:
        outer_procs = effective_cpus

    if n_cores is not None:
        outer_procs = min(int(n_cores), outer_procs)

    outer_procs = max(1, min(len(inputs), outer_procs))

    print(f"Benchmark {benchmark}: {len(inputs)} total alignment tasks")
    if saw_cpu_parflex:
        print(f"Estimated `cpu_parflex` inner pool max workers: {max_inner_workers}")
    print(f"Outer pool size (threads): {outer_procs}")

    # Use a thread pool for the outer scheduler so `cpu_parflex` can spawn
    # its own worker processes without hitting daemonic-process restrictions.
    with mp_pool.ThreadPool(processes=outer_procs) as pool:
        for _ in tqdm(pool.imap_unordered(_align_system_task, inputs), total=len(inputs)):
            pass

    return

In [17]:
def run_benchmark(benchmark, featdir1, featdir2, parts, outdir, queryid):
    featfile1 = (featdir1 / parts[0]).with_suffix('.npy')
    featfile2 = (featdir2 / parts[1]).with_suffix('.npy')

    F1 = np.load(featfile1)
    F2 = np.load(featfile2)
        
    # run all baselines
    for system in SYSTEMS:
        if system == 'sparse_parflex':
            for L in SPARSE_PARFLEX_L_VALUES:
                outfile = get_outfile(outdir, benchmark, system, queryid, L=L)
                if not os.path.isfile(outfile):
                    align_system(system, F1, F2, outfile, L)
        else:
            outfile = get_outfile(outdir, benchmark, system, queryid)
            if not os.path.isfile(outfile):
                align_system(system, F1, F2, outfile, None)

In [18]:
outdir = Path(f'experiments_{DATASET}/{VERSION}')
run_all_benchmarks(outdir)

  0%|          | 0/16 [00:00<?, ?it/s]

Benchmark matching: 5 total alignment tasks
Outer pool size (threads): 5


  6%|▋         | 1/16 [00:25<06:28, 25.90s/it]

Benchmark subseq_20: 10 total alignment tasks
Outer pool size (threads): 10


 12%|█▎        | 2/16 [00:33<03:32, 15.18s/it]

...

Benchmark partialOverlap: 10 total alignment tasks
Outer pool size (threads): 10


 44%|████▍     | 7/16 [01:38<02:05, 13.99s/it]

Benchmark pre_5: 10 total alignment tasks
Outer pool size (threads): 10


 44%|████▍     | 7/16 [01:59<02:33, 17.00s/it]


ValueError: Stage 2: No valid endpoint found on global top/right edges.